# The Alpha Transform

## Stability Inversion for Fixed-Point Iteration

**Author:** Keenan M Stone  
**Origin:** May 2017 (original Petrification scripts)  
**This notebook:** April 2026  
**Status:** Self-contained review of the alpha transform, its theory, applications, and placement in the literature

---

### Purpose

This notebook develops the **alpha-transform** — the map

$$g(x) = \alpha f(x) + (1-\alpha)x$$

as a tool for controlling fixed-point stability. We:

1. State and prove the core theorems on stability inversion
2. Develop the position-dependent generalization $\alpha(x)$
3. Connect to the literature (Krasnoselskii-Mann, SOR, Anderson mixing)
4. Demonstrate applications on discrete maps (logistic, Migdal-Kadanoff)
5. Connect fixed points to eigenstates in projective Hilbert space
6. Assess what, if anything, is novel vs. well-known

The alpha transform likely fits into a broader class of **concavity-inverting transforms** — it modifies the curvature landscape of an iteration map to swap attractors and repulsors. Whether this framing adds value beyond existing theory is an open question we address honestly.

## 1. Literature Review: Fixed Points and Eigenstates

### 1.1 Fixed Points in Classical Dynamical Systems

The theory of fixed-point iteration is foundational to analysis. A map $f: X \to X$ on a metric space has a fixed point $x^*$ where $f(x^*) = x^*$. The central results:

- **Banach contraction mapping theorem** (1922): If $f$ is a contraction ($|f(x) - f(y)| \le L|x-y|$ with $L < 1$), the iteration $x_{n+1} = f(x_n)$ converges to a unique fixed point from any initial condition.
- **Local stability**: For $C^1$ maps on $\mathbb{R}$, a fixed point is stable (attracting) if $|f'(x^*)| < 1$ and unstable (repelling) if $|f'(x^*)| > 1$.
- **Bifurcation theory**: As parameters vary, fixed points can change stability, split, or disappear. The period-doubling cascade in the logistic map $f(x) = ax(1-x)$ is the prototypical example [Strogatz, Ch. 10].

### 1.2 Fixed Points in Quantum Mechanics — The Projective Perspective

The eigenvalue equation $\hat{A}|\psi\rangle = \lambda|\psi\rangle$ says: an operator acts on a state and returns *the same state* up to scaling. In **projective Hilbert space** $\mathbb{P}(\mathcal{H})$ — where $|\psi\rangle \sim c|\psi\rangle$ — this is exactly a **fixed point** of the projective action:

$$\Pi_A : [|\psi\rangle] \mapsto \left[\frac{\hat{A}|\psi\rangle}{\|\hat{A}|\psi\rangle\|}\right]$$

Then $[|\psi_k\rangle]$ is a fixed point of $\Pi_A$ if and only if $|\psi_k\rangle$ is an eigenstate of $\hat{A}$.

| Fixed-point dynamics | Quantum eigenvalue problem |
|---|---|
| $f(x^*) = x^*$ | $\hat{A}|\psi\rangle = \lambda|\psi\rangle$ |
| $|f'(x^*)| < 1$ : stable (attracting) | $|\lambda|$ largest: power method converges |
| $|f'(x^*)| > 1$ : unstable (repelling) | $|\lambda|$ not largest: power method diverges |
| Alpha-transform $g = \alpha f + (1-\alpha)x$ | Relaxed iteration / spectral shift |
| Bifurcation in parameter $a$ | Spectral phase transitions |

**This is not just an analogy — it is the mathematical content of power iteration** [Trefethen & Bau, Ch. 25].

Several concrete algorithms already embody this connection:

- **Power iteration** converges to the eigenstate with largest $|\lambda|$ — the most "stable" fixed point of $\Pi_A$
- **Inverse power iteration with shift** $(\hat{A} - \sigma I)^{-1}$ converges to the eigenstate *nearest* $\sigma$ — fixed-point iteration with a *modified stability landscape*
- **Spectral shooting methods** find eigenvalues as zeros of a boundary mismatch — root-finding that is itself a fixed-point problem

### 1.3 Renormalization Group as Fixed-Point Iteration

Wilson's renormalization group [Wilson 1975] treats phase transitions as fixed points of block-spin maps. Migdal-Kadanoff approximations [Migdal 1975, Kadanoff 1966] reduce lattice gauge theory to iterated maps. The RG flow $\beta$-function is a continuous-time analog of what we study here discretely.

### 1.4 Relaxation Methods in Numerical Analysis

The alpha-transform is closely related to established techniques:

- **Krasnoselskii-Mann iteration** [Krasnoselskii 1955]: $x_{n+1} = (1-\theta)x_n + \theta T(x_n)$ for nonexpansive $T$. Setting $\theta = \alpha$ recovers our transform.
- **Successive Over-Relaxation (SOR)**: In numerical linear algebra, the SOR method uses a mixing parameter $\omega$ to accelerate Gauss-Seidel iteration. The optimal $\omega$ depends on the spectral radius of the iteration matrix.
- **Anderson mixing / DIIS**: More sophisticated self-consistent-field methods that use a history of iterates to extrapolate. These are nonlinear generalizations.

**The honest question:** Is the alpha-transform anything more than a relabeling of these known techniques? The answer may be that the value lies not in the transform itself but in the systematic framework for choosing $\alpha$ — particularly the position-dependent $\alpha(x)$ generalization.

### 1.5 References

1. **Strogatz, S.H.** *Nonlinear Dynamics and Chaos.* 2nd ed., CRC Press, 2015.
2. **Trefethen, L.N. and Bau, D.** *Numerical Linear Algebra.* SIAM, 1997.
3. **Wilson, K.G.** "The renormalization group." *Rev. Mod. Phys.* 47.4 (1975): 773.
4. **Kadanoff, L.P.** "Scaling laws for Ising models near $T_c$." *Physics* 2.6 (1966): 263.
5. **Migdal, A.A.** "Recursion equations in gauge theories." *Sov. Phys. JETP* 42.3 (1975): 413.
6. **Krasnoselskii, M.A.** "Two remarks on the method of successive approximations." *Uspekhi Mat. Nauk* 10.1 (1955): 123.
7. **Güttel, S. and Tisseur, F.** "The nonlinear eigenvalue problem." *Acta Numerica* 26 (2017): 1–94.
8. **Stone, K.M.** "Petrification scripts." (2017, unpublished).

## 2. The Alpha Transform: Theory

### 2.1 Definition and Fixed-Point Preservation

Let $f: \mathbb{R} \to \mathbb{R}$ be a $C^1$ map with fixed points $\{x_i^*\}$ where $f(x_i^*) = x_i^*$. Define:

$$g(x) = \alpha f(x) + (1 - \alpha) x$$

Since $f(x_i^*) = x_i^*$, we have $g(x_i^*) = \alpha x_i^* + (1-\alpha) x_i^* = x_i^*$, so **$g$ has exactly the same fixed points as $f$** for any $\alpha$.

### 2.2 Proofs

**Proposition 1 (Transformed slope).** At any fixed point $x^*$:
$$g'(x^*) = \alpha f'(x^*) + (1 - \alpha) = 1 + \alpha\bigl(f'(x^*) - 1\bigr)$$

*Proof.* $g'(x) = \alpha f'(x) + (1 - \alpha)$. Evaluate at $x = x^*$. $\square$

Writing $\lambda = f'(x^*)$, the transformed slope is $g'(x^*) = 1 + \alpha(\lambda - 1)$.

---

**Proposition 2 (Stability interval).** The fixed point $x^*$ is stable under $g$ (i.e., $|g'(x^*)| < 1$) if and only if:
$$-\frac{2}{\lambda - 1} < \alpha < 0 \quad \text{when } \lambda > 1$$
$$0 < \alpha < \frac{2}{1 - \lambda} \quad \text{when } \lambda < -1$$

*Proof.* We need $|1 + \alpha(\lambda-1)| < 1$. Expanding:
$$-1 < 1 + \alpha(\lambda - 1) < 1$$
$$-2 < \alpha(\lambda - 1) < 0$$
When $\lambda > 1$: $\lambda - 1 > 0$, so $-2/(\lambda-1) < \alpha < 0$.  
When $\lambda < -1$: $\lambda - 1 < 0$, dividing reverses inequalities, giving $0 < \alpha < 2/(1-\lambda)$. $\square$

---

**Proposition 3 (Optimal alpha).** The value $\alpha^* = 1/(1 - \lambda)$ **zeroes the transformed slope** ($g'(x^*) = 0$), giving the fastest possible local convergence.

*Proof.* $g'(x^*) = 1 + \alpha(\lambda - 1) = 1 + \frac{\lambda - 1}{1 - \lambda} = 1 - 1 = 0$. $\square$

Note: $\alpha^*$ is the midpoint of the stability interval.

---

**Proposition 4 (Obstruction to global inversion).** If $f$ has both an unstable fixed point with $\lambda_i > 1$ and another with $\lambda_j < -1$, then **no single $\alpha$ can stabilize both simultaneously**.

*Proof.* Stabilizing $\lambda_i > 1$ requires $\alpha < 0$. Stabilizing $\lambda_j < -1$ requires $\alpha > 0$. These are disjoint. $\square$

**Example:** For the logistic map at $a = 4$, the fixed points are $x_0^* = 0$ with $f'(0) = 4 > 1$, and $x_1^* = 3/4$ with $f'(3/4) = -2 < -1$. No single alpha stabilizes both.

---

**Proposition 5 (Same-sign stabilization).** If all unstable fixed points have slopes of the same sign (all $\lambda_i > 1$ or all $\lambda_i < -1$), then $\alpha^* = 1/(1 - \lambda_{\max})$ stabilizes **all** of them simultaneously, and furthermore **destabilizes all previously stable fixed points**.

*Proof.* Consider the case $\lambda_i > 1$ for all unstable $x_i^*$. Take $\lambda_{\max} = \max_i \lambda_i$ and $\alpha^* = 1/(1-\lambda_{\max}) < 0$.

For any unstable $x_i^*$: Since $1 \leq \lambda_i \leq \lambda_{\max}$, we have $0 \leq (\lambda_i-1)/(\lambda_{\max}-1) \leq 1$, so $g'(x_i^*) \in [-1, 0) \subset (-1,1)$. Stable. ✓

For any stable $x_j^*$ with $|\lambda_j| < 1$: Since $\lambda_j < 1$ and $1-\lambda_{\max} < 0$, the correction term is positive: $g'(x_j^*) > 1$. Unstable. ✓ $\square$

### 2.3 Connection to Curvature

To find $\lambda_{\max} = \max_x f'(x)$, we solve $f''(x) = 0$ for critical points of $f'$. For polynomials, these are inflection points, not points of maximum curvature $|f''|$. The coincidence for simple maps (the logistic map has constant $f''$) may have caused earlier confusion between "inverse of slope at max curvature" and the correct formula.

### 2.4 Operator Structure

Using operator notation $\Gamma_\alpha f(x) = \alpha f(x) + (1-\alpha)x$:
- $\Gamma_1 = \text{id}$ (identity)
- $\Gamma_\alpha \Gamma_\beta = \Gamma_{\alpha\beta}$ (successive applications compose multiplicatively)
- $\Gamma_{\alpha^{-1}} = \Gamma_\alpha^{-1}$ (invertible)
- Isomorphic to $(\mathbb{R}_{>0}, \cdot)$ under $\alpha \mapsto \Gamma_\alpha$

## 3. The Position-Dependent Generalization: $\alpha(x)$

### 3.1 Theory

Consider $\alpha(x)$:
$$g(x) = \alpha(x) f(x) + (1 - \alpha(x)) x$$

**Fixed points are still preserved:** if $f(x^*) = x^*$, then $g(x^*) = \alpha(x^*) x^* + (1-\alpha(x^*)) x^* = x^*$ for any $\alpha(x^*)$.

**Derivative at a fixed point:**
$$g'(x^*) = \alpha'(x^*)\underbrace{(f(x^*) - x^*)}_{= 0} + \alpha(x^*) f'(x^*) + (1 - \alpha(x^*)) = 1 + \alpha(x^*)(f'(x^*) - 1)$$

The $\alpha'$ term vanishes because $f(x^*) = x^*$. So **the stability at each fixed point depends only on $\alpha(x^*)$**, not on $\alpha'(x^*)$.

### 3.2 Overcoming Proposition 4

The obstruction was that no single constant $\alpha$ can stabilize fixed points with slopes of both signs. With $\alpha(x)$, each fixed point gets its own $\alpha$ value — the choice $\alpha(x^*) = 1/(1 - f'(x^*))$ zeroes $g'$ at $x^*$ automatically, for each fixed point independently.

### 3.3 Gauge Structure

With position-dependent $\alpha(x)$, the global group structure ($\Gamma_\alpha \Gamma_\beta = \Gamma_{\alpha\beta}$) becomes a **gauge transformation** — the parameter varies over the base space, analogous to how gauge fields in physics assign different group elements to different spacetime points.

### 3.4 The Tradeoff

The formula $\alpha(x) = 1/(1-f'(a,x))$ has a singularity where $f'(a,x) = 1$, creating a "wall" in the iteration landscape. Trajectories passing near this singularity get disrupted. Capping $|\alpha| \leq C$ prevents divergence but fragments the basin of attraction. This means:

- **$\alpha(x)$ is locally superior** (superlinear convergence near each fixed point)
- **$\alpha(x)$ is globally inferior** (fragmented basin compared to constant $\alpha$)

This tradeoff is the central practical limitation.

## 4. Placing in the Literature: Concavity-Inverting Transforms

The alpha-transform alters the curvature landscape of an iteration map. At its core:

$$g''(x) = \alpha f''(x)$$

so for $\alpha < 0$, the concavity of $g$ is flipped relative to $f$. This inverts the relationship between curvature and fixed-point stability: what was concave-down (creating a stable attracting fixed point with $f' < 0$) becomes concave-up (repelling), and vice versa.

**How this relates to known techniques:**

| Technique | What it does | $\alpha$-transform equivalent |
|-----------|-------------|------------------------------|
| SOR ($\omega$-relaxation) | Mixes update with current value | Constant $\alpha$ |
| Krasnoselskii-Mann | $(1-\theta)x + \theta Tx$ | $\alpha = \theta$ |
| Anderson mixing | Uses history to extrapolate | $\alpha$ from past iterates |
| Broyden mixing (DMFT) | Quasi-Newton with rank-1 updates | Adaptive matrix $\alpha$ |
| **$\alpha(x)$ transform** | Position-dependent relaxation | Independent per fixed point |

**What the $\alpha(x)$ generalization adds beyond Krasnoselskii-Mann:**

1. The key result that $g'(x^*) = 1 + \alpha(x^*)(f'(x^*) - 1)$ — where the $\alpha'$ term vanishes — means position-dependent relaxation gives **independent local control** at each fixed point. This is a clean result that is implicitly known in the preconditioning literature but rarely stated in this explicit form.

2. The obstruction theorem (Prop. 4) and its resolution via $\alpha(x)$ is a useful pedagogical contribution.

3. The gauge-theoretic framing ($\alpha(x)$ as a gauge parameter varying over phase space) is suggestive but currently just language, not yet a source of new results.

**Honest assessment:** The alpha-transform is likely not a "new discovery" — the ideas are scattered across the fixed-point theory, numerical analysis, and preconditioning literature. The potential contribution is the **systematic synthesis**: stating the stability inversion theorems cleanly, proving the $\alpha'$ cancellation for position-dependent $\alpha$, identifying the obstruction and its resolution, and connecting all of this to eigenvalue problems via the projective fixed-point perspective.

## 5. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh

from petrification.maps import logistic, migdal_kadanoff
from petrification.transforms import (alpha_transform, compute_optimal_alpha,
                                       make_alpha_func, optimal_alpha_at_x)
from petrification.bifurcation import compute_bifurcation, compute_bifurcation_transformed
from petrification.iteration import iterate, iterate_transformed, cobweb_data, find_fixed_points
from petrification.potentials import harmonic
from petrification.quantum import (
    discretize_hamiltonian, solve_eigenstates,
    inverse_power_iteration, projective_power_iteration, alpha_eigenstate_scan,
)

%matplotlib inline
plt.rcParams.update({'figure.figsize': (12, 8), 'font.size': 12})
print('All imports successful.')

## 6. Demonstrations on Discrete Maps

### 6.1 Cobweb Diagrams: Stabilizing the Logistic Map

For the logistic map $f(x) = ax(1-x)$ at $a = 4.0$, the fixed point at $x^* = 3/4$ is **unstable** ($|f'(x^*)| = |2 - a| = 2 > 1$). The alpha-transform stabilizes it.

In [ ]:
a = 4.0
x0 = 0.7

# Compute optimal alpha
alpha, info = compute_optimal_alpha(logistic, a, x_domain=(0.0, 1.0))
print(f"Optimal alpha = {alpha:.4f}")
print(f"  Method: {info['method']}, df_max = {info['df_max']}, df_min = {info['df_min']}")

# Cobweb data
Ix_orig, Iy_orig = cobweb_data(logistic, a, x0, n_iter=20)
Ix_trans, Iy_trans = cobweb_data(logistic, a, x0, n_iter=20, alpha=alpha)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
lx = np.linspace(-0.1, 1.1, 500)

ax1.plot(lx, logistic(a, lx), 'b-', lw=2, label=r'$f(x) = ax(1-x)$')
ax1.plot(lx, lx, 'k-', lw=1)
ax1.plot(Ix_orig, Iy_orig, 'r-', lw=0.8, alpha=0.7)
ax1.set_xlabel(r'$x_n$'); ax1.set_ylabel(r'$x_{n+1}$')
ax1.set_title(f'Original: $a={a}$, diverges from fixed point')
ax1.set_xlim(-0.1, 1.1); ax1.set_ylim(-0.1, 1.1)
ax1.legend()

ly_t = alpha * logistic(a, lx) + (1 - alpha) * lx
ax2.plot(lx, ly_t, 'b-', lw=2, label=rf'$g(x) = \alpha f + (1-\alpha)x$')
ax2.plot(lx, lx, 'k-', lw=1)
ax2.plot(Ix_trans, Iy_trans, 'g-', lw=0.8, alpha=0.7)
ax2.set_xlabel(r'$x_n$'); ax2.set_ylabel(r'$x_{n+1}$')
ax2.set_title(rf'Alpha-transformed: $\alpha={alpha:.3f}$, converges to fixed point')
ax2.set_xlim(-0.1, 1.1); ax2.set_ylim(-0.1, 1.1)
ax2.legend()

plt.tight_layout()
plt.show()

### 6.2 Numerical Verification of the Stability Theorems

In [ ]:
print("=" * 72)
print("VERIFICATION: Alpha-Transform Stability for Logistic Map f(x)=ax(1-x)")
print("=" * 72)

for a_test in [2.5, 3.2, 3.83, 4.0]:
    x_fp = [0.0, 1 - 1/a_test]
    slopes = [a_test, 2 - a_test]
    
    print(f"\n--- a = {a_test} ---")
    for x_star, lam in zip(x_fp, slopes):
        alpha_opt = 1.0 / (1.0 - lam) if lam != 1 else float('inf')
        alpha_bdy = 2.0 / (1.0 - lam) if lam != 1 else float('inf')
        g_prime_opt = 1 + alpha_opt * (lam - 1) if np.isfinite(alpha_opt) else float('inf')
        g_prime_bdy = 1 + alpha_bdy * (lam - 1) if np.isfinite(alpha_bdy) else float('inf')
        stable_orig = "|\u03bb|<1 \u2713" if abs(lam) < 1 else "|\u03bb|\u22651 \u2717"
        
        print(f"  x* = {x_star:.4f}, f'(x*) = {lam:+.2f}  {stable_orig}")
        print(f"    \u03b1_optimal = {alpha_opt:+.4f} \u2192 g'(x*) = {g_prime_opt:+.4f}")
        print(f"    \u03b1_boundary = {alpha_bdy:+.4f} \u2192 g'(x*) = {g_prime_bdy:+.4f}")

print("\n" + "=" * 72)
print("Proposition 4 check: at a=4, x=0 needs \u03b1<0, x=3/4 needs \u03b1>0")
print("\u2192 No single \u03b1 stabilizes both.")
print("=" * 72)

# Convergence comparison: optimal vs boundary alpha at a=3.2
print("\n--- Convergence comparison at a=3.2, targeting x*=0.6875 ---")
a_demo = 3.2
x_star_demo = 1 - 1/a_demo
lam_demo = 2 - a_demo
alpha_opt_demo = 1.0 / (1.0 - lam_demo)
alpha_bdy_demo = 2.0 / (1.0 - lam_demo)

x_opt, x_bdy = 0.5, 0.5
print(f"  \u03b1_optimal = {alpha_opt_demo:.4f}, \u03b1_boundary = {alpha_bdy_demo:.4f}")
print(f"  {'iter':>4}  {'x_optimal':>12}  {'x_boundary':>12}  {'err_opt':>10}  {'err_bdy':>10}")
for n in range(12):
    err_o = abs(x_opt - x_star_demo)
    err_b = abs(x_bdy - x_star_demo)
    print(f"  {n:4d}  {x_opt:12.8f}  {x_bdy:12.8f}  {err_o:10.2e}  {err_b:10.2e}")
    x_opt = alpha_opt_demo * logistic(a_demo, x_opt) + (1 - alpha_opt_demo) * x_opt
    x_bdy = alpha_bdy_demo * logistic(a_demo, x_bdy) + (1 - alpha_bdy_demo) * x_bdy

print(f"\nOptimal converges superlinearly (g'=0 \u2192 error squares each step).")
print(f"Boundary oscillates with |g'|=1 \u2014 does NOT converge.")

### 6.3 Bifurcation Diagrams: Original vs. Transformed

In [ ]:
a_range = np.linspace(2.5, 4.0, 400)
x0_samples = np.linspace(0.1, 0.9, 30)
alpha_bif = -1.0

a_orig, x_orig = compute_bifurcation(logistic, a_range, x0_samples,
                                      n_iter=500, n_discard=400)
a_trans, x_trans = compute_bifurcation_transformed(logistic, alpha_bif, a_range,
                                                    x0_samples, n_iter=500, n_discard=400)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.scatter(a_orig, x_orig, s=0.1, c='black', alpha=0.5)
ax1.set_xlabel('Parameter $a$'); ax1.set_ylabel('$x$')
ax1.set_title('Original: $f(x) = ax(1-x)$')

ax2.scatter(a_trans, x_trans, s=0.1, c='black', alpha=0.5)
ax2.set_xlabel('Parameter $a$'); ax2.set_ylabel('$x$')
ax2.set_title(rf'Transformed: $g = \alpha f + (1-\alpha)x$, $\alpha = {alpha_bif}$')

plt.tight_layout()
plt.show()

### 6.4 Position-Dependent $\alpha(x)$: Simultaneous Stabilization

Here we demonstrate that $\alpha(x) = 1/(1-f'(a,x))$ overcomes the Proposition 4 obstruction, simultaneously stabilizing both fixed points of the logistic map at $a = 4$.

In [ ]:
import importlib
import petrification.transforms, petrification.iteration
importlib.reload(petrification.transforms)
importlib.reload(petrification.iteration)
from petrification.transforms import make_alpha_func
from petrification.iteration import iterate_transformed

def logistic_prime(a, x):
    return a * (1 - 2*x)

a_test = 4.0
x_fp0 = 0.0
x_fp1 = 1 - 1/a_test  # = 0.75

alpha_func = make_alpha_func(logistic_prime, a_test, smooth=True, cap=5.0)

# Verify: both fixed points get g'(x*) = 0
for x_fp, label in [(x_fp0, 'x*=0'), (x_fp1, 'x*=0.75')]:
    lam = logistic_prime(a_test, x_fp)
    a_val = alpha_func(x_fp)
    g_prime = 1 + a_val * (lam - 1)
    print(f"{label}: f'={lam:+.2f}, \u03b1(x*)={a_val:+.4f}, g'(x*)={g_prime:+.6f}")

# Compare convergence: constant α vs α(x)
alpha_const = 1.0 / (1.0 - logistic_prime(a_test, x_fp1))  # optimal for x*=0.75
n_iter = 60
x0_near = 0.8

traj_none = iterate_transformed(logistic, 1.0, a_test, x0_near, n_iter)
traj_const = iterate_transformed(logistic, alpha_const, a_test, x0_near, n_iter)
traj_func = iterate_transformed(logistic, alpha_func, a_test, x0_near, n_iter)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, traj, title in [
    (axes[0], traj_none, f'No transform (\u03b1=1)'),
    (axes[1], traj_const, f'Constant \u03b1={alpha_const:.4f}'),
    (axes[2], traj_func, '\u03b1(x) = 1/(1-f\'(a,x))'),
]:
    ax.plot(traj, 'b-', lw=0.8, alpha=0.7)
    ax.axhline(x_fp1, color='red', ls='--', alpha=0.5)
    ax.set_title(title)
    ax.set_ylabel('$x_n$')
    ax.set_xlabel('Iteration')
    ax.set_ylim(-0.5, 1.5)

plt.suptitle('Proposition 4 Obstruction Overcome by \u03b1(x)', fontsize=13)
plt.tight_layout()
plt.show()

# Convergence rates across Feigenbaum cascade
a_cascade = np.linspace(2.5, 4.0, 60)
conv_rate_const = np.zeros(len(a_cascade))
conv_rate_func = np.zeros(len(a_cascade))

for j, a_v in enumerate(a_cascade):
    x_fp = 1 - 1/a_v if a_v > 1 else 0
    af = make_alpha_func(logistic_prime, a_v, smooth=True, cap=5.0)
    ac = 1.0 / (1.0 - logistic_prime(a_v, x_fp)) if abs(1 - logistic_prime(a_v, x_fp)) > 0.01 else 1.0
    
    n_conv_c, n_conv_f = 0, 0
    for x0 in np.linspace(0.01, 0.99, 20):
        tc = iterate_transformed(logistic, ac, a_v, x0, 100)
        tf = iterate_transformed(logistic, af, a_v, x0, 100)
        if abs(tc[-1] - x_fp) < 1e-6: n_conv_c += 1
        if abs(tf[-1] - x_fp) < 1e-6: n_conv_f += 1
    conv_rate_const[j] = n_conv_c / 20
    conv_rate_func[j] = n_conv_f / 20

fig, ax = plt.subplots(1, 1, figsize=(12, 5))
ax.plot(a_cascade, conv_rate_const, 'b-o', ms=4, label='Constant \u03b1 (optimal at x*)')
ax.plot(a_cascade, conv_rate_func, 'r-s', ms=4, label='\u03b1(x) = 1/(1-f\'(a,x))')
ax.axvline(3.0, color='gray', alpha=0.3, ls='--')
ax.axvline(3.449, color='gray', alpha=0.3, ls=':')
ax.axvline(3.5699, color='gray', alpha=0.3, ls='-.')
ax.set_xlabel('Parameter $a$')
ax.set_ylabel('Convergence fraction (20 ICs)')
ax.set_title('\u03b1(x) vs constant \u03b1: local superiority vs global basin fragmentation')
ax.legend()
ax.set_ylim(-0.05, 1.1)
plt.tight_layout()
plt.show()

## 7. Application to Quantum Eigenvalue Problems

### 7.1 Eigenstates as Projective Fixed Points

We demonstrate the eigenvalue-fixed-point connection directly: power iteration converges to the dominant eigenstate, and inverse power iteration with different shifts selects different eigenstates — exactly like the alpha-transform selects different fixed points.

In [ ]:
# Harmonic oscillator eigenvalues via matrix diagonalization
N = 500
x_grid = np.linspace(-8, 8, N)

eigenvalues, eigenvectors = solve_eigenstates(harmonic, x_grid, n_states=8)
exact = np.arange(8) + 0.5

print("Harmonic oscillator eigenvalues:")
print(f"{'n':>3}  {'Computed':>12}  {'Exact':>8}  {'Error':>10}")
for i in range(8):
    print(f"{i:3d}  {eigenvalues[i]:12.6f}  {exact[i]:8.1f}  {abs(eigenvalues[i] - exact[i]):10.2e}")

In [ ]:
# Inverse power iteration: convergence to different eigenstates via shift
H = discretize_hamiltonian(harmonic, x_grid)
evals_full = eigh(H, eigvals_only=True)

fig, ax = plt.subplots(figsize=(10, 6))
shifts = [0.4, 1.6, 2.4, 3.6, 4.6]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

for sigma, color in zip(shifts, colors):
    rq = inverse_power_iteration(H, sigma, n_iter=40)
    nearest_idx = np.argmin(np.abs(evals_full - sigma))
    nearest_E = evals_full[nearest_idx]
    ax.plot(rq, color=color, lw=2,
            label=rf'$\sigma={sigma:.1f} \to E_{nearest_idx}={nearest_E:.2f}$')

for i in range(5):
    ax.axhline(evals_full[i], color='gray', alpha=0.3, lw=0.8)

ax.set_xlabel('Iteration $n$')
ax.set_ylabel(r'Rayleigh quotient $\langle x|H|x\rangle$')
ax.set_title('Inverse power iteration = fixed-point convergence to nearest eigenstate\n'
             'Each shift selects a different attracting fixed point')
ax.legend(fontsize=10)
ax.set_ylim(0, 5.5)
plt.tight_layout()
plt.show()

In [ ]:
# Alpha-scan: which eigenstate does the iteration converge to for each alpha?
N_small = 60
x_small = np.linspace(-5, 5, N_small)
H_small = discretize_hamiltonian(harmonic, x_small)

alphas = np.linspace(-2.0, 2.0, 500)
rng = np.random.default_rng(7)
x0 = rng.standard_normal(N_small)

dom, quality, ev_small = alpha_eigenstate_scan(H_small, x0, alphas, n_iter=500)

fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(alphas, dom, c=quality, cmap='viridis', s=3, vmin=0, vmax=1)
ax.set_xlabel(r'$\alpha$')
ax.set_ylabel('Dominant eigenstate index')
ax.set_title(r'$\alpha$-transform selects different eigenstates (color = convergence quality)')
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label(r'$|\langle\psi_k|x\rangle|^2$')
plt.tight_layout()
plt.show()

## 8. Assessment and Further Considerations

### 8.1 What Is Established

- **Props 1-5** are clean, provable results about the alpha-transform's effect on fixed-point stability
- The **$\alpha(x)$ generalization** overcomes the constant-$\alpha$ obstruction and provides independent local control
- The **projective fixed-point interpretation** of eigenstates is mathematically rigorous (not novel, but useful framing)
- The alpha-transform is a member of the **Krasnoselskii-Mann family** of relaxation methods

### 8.2 What Might Be Worth Developing

1. **The $\alpha'$ cancellation theorem** (that $g'(x^*)$ depends only on $\alpha(x^*)$, not $\alpha'(x^*)$) is implicit in the preconditioning literature but rarely stated this cleanly. Worth formalizing.

2. **The concavity-inversion perspective** ($g'' = \alpha f''$, so $\alpha < 0$ flips curvature) may have applications beyond iteration — e.g., in optimization where convexity/concavity of the objective matters.

3. **Basin fragmentation from $\alpha(x)$**: The singularity at $f'(a,x) = 1$ is where $\alpha(x) \to \infty$. Regularizing this (e.g., $\alpha(x) = 1/(1-f'(a,x) + \epsilon)$) might preserve the local superlinear convergence while maintaining global basin connectivity.

4. **Matrix-valued $\alpha$**: For multi-dimensional fixed-point problems ($\mathbf{x} = \mathbf{f}(\mathbf{x})$), $\alpha$ becomes a matrix. The optimal $\alpha = (I - J_f(x^*))^{-1}$ is exactly the Newton step direction. This connects the alpha-transform to Newton's method in a precise way.

5. **Application to Dyson equations**: See the crossover notebook for the application of constant $\alpha \approx 0.5$ to the Dyson equation, where it provides universal convergence across all coupling strengths.

### 8.3 Prior Art Check

Before claiming any novelty, verify:
- [ ] Does the preconditioning literature already state the $\alpha'$ cancellation for position-dependent relaxation?
- [ ] Is the "concavity inversion" framing ($g'' = \alpha f''$) used anywhere in optimization theory?
- [ ] Are there results connecting gauge transformations on iteration parameters to fixed-point stability?
- [ ] Is the basin fragmentation from singular $\alpha(x)$ studied in the dynamical systems literature?